# CNN V5.1 P5 runner (Google Colab)

This notebook is the Colab adapter for P5 under the O0-A5 1 GiB MUSAN selection and 105x B0 clip cap. It does not train a model or open test/OOD waveforms. Keep the data and outputs in your private Google Drive.

> **Current gate status:** Historical corpus-007/split-007 failed B0 under the 20x clip cap and is ineligible for P5. O0-A5 authorizes a new split-008 attempt under a 105x clip cap while retaining the <=1 GiB byte limit. The split-008 P4 attempt has now passed R-LED, B0, B1, B2, and B3_B4 under the 105x cap. P5 is NOT_RUN. Upload the matching `p4_summary.json` and `weight_fit.json` from attempt `v51-budget-1g-a5`; the runner verifies their membership and contract bindings before starting.

## Drive layout

Upload the `study-v5-1` workspace code (including `src/`, `config/`, `scripts/`, `docs/v5.1/DATA_PREPARATION_CNN_V5_1_CONTRACT.json`, and `requirements-v51.txt`) and these inputs under one private Drive folder. The admitted source data may have redistribution restrictions; do not share this Drive folder publicly.

```text
MyDrive/CNN_V5_1_P5/
  study-v5-1/                         # code workspace
  frozen/p3-008/                       # new O0-A5 split-008 freeze; generate only after P1-P3
  p4-008/                              # copy p4_summary.json and weight_fit.json from local O0-A5 P4 PASS
  inventory_v51_007.csv               # byte-identical corpus-007 inventory
  musan_budget_selection.csv          # unchanged O0-A4 complete selection/exclusion manifest
  sources/ae/                          # registered AE source root
  sources/dd/                          # registered DD source root
  sources/itu/                         # registered ITU-ARIS source root
  sources/musan/                       # full registered MUSAN tree; only selected train/val IDs are read
  outputs/                             # new P5 attempt output is created here
```

Use the O0-A5 split-008 freeze and its matching P1/P2/P3 artifacts. The O0-A5 P4 attempt passed R-LED, B0, B1, B2, and B3_B4 at 105x; copy its matching `p4_summary.json` and `weight_fit.json` into `p4-008/`. The runner verifies those identities. The earlier split-007 P4 failed at 20x and is never eligible. corpus-005/split-005 and corpus-006/split-006 remain historical. Preserve inventoried relative paths. The input membership totals 1,073,741,524 bytes (300 bytes below 1 GiB). The contract must match the SHA-256 bound in `FROZEN.json`. P5 uses CPU; its deterministic A-chk pair ranking is CPU-bound.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import datetime, subprocess, sys, uuid

DRIVE_ROOT = Path('/content/drive/MyDrive/CNN_V5_1_P5')
STUDY = DRIVE_ROOT / 'study-v5-1'
FROZEN = DRIVE_ROOT / 'frozen/p3-008'
P4 = DRIVE_ROOT / 'p4-008'
INVENTORY = DRIVE_ROOT / 'inventory_v51_007.csv'
SELECTION = DRIVE_ROOT / 'musan_budget_selection.csv'
SOURCES = {
    'ae': DRIVE_ROOT / 'sources/ae',
    'dd': DRIVE_ROOT / 'sources/dd',
    'itu': DRIVE_ROOT / 'sources/itu',
    'musan': DRIVE_ROOT / 'sources/musan',
}
OUT = DRIVE_ROOT / 'outputs' / ('p5-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])

required = [STUDY / 'scripts/run_cnn_v51_p5_colab.py', STUDY / 'requirements-v51.txt',
            STUDY / 'config/v51.yaml',
            STUDY / 'docs/v5.1/DATA_PREPARATION_CNN_V5_1_CONTRACT.json',
            FROZEN / 'FROZEN.json', SELECTION,
            P4 / 'p4_summary.json', P4 / 'weight_fit.json', INVENTORY]
missing = [str(p) for p in required if not p.exists()] + [str(p) for p in SOURCES.values() if not p.is_dir()]
if missing:
    raise FileNotFoundError('Missing required Drive inputs:\n' + '\n'.join(missing))

import hashlib, json
frozen_record = json.loads((FROZEN / 'FROZEN.json').read_text(encoding='utf-8'))
contract_path = STUDY / 'docs/v5.1/DATA_PREPARATION_CNN_V5_1_CONTRACT.json'
contract_sha256 = hashlib.sha256(contract_path.read_bytes()).hexdigest()
expected_contract_sha256 = frozen_record.get('contract_sha256')
print('Contract SHA-256:', contract_sha256)
if contract_sha256 != expected_contract_sha256:
    raise ValueError(f'Contract hash mismatch: expected {expected_contract_sha256}, got {contract_sha256}')
OUT.parent.mkdir(parents=True, exist_ok=True)
print('New output directory:', OUT)
print('This attempt will not overwrite existing outputs.')

In [ ]:
# Install the exact preparation dependency pins from the uploaded workspace.
!{sys.executable} -m pip install -r "{STUDY / 'requirements-v51.txt'}"

In [ ]:
# Cheap preflight: resolve every train/validation source file before the expensive A-chk ranking.
import csv, hashlib, numpy as np
sys.path.insert(0, str(STUDY / 'src'))
from dronedet.cnn_v51.config import load_v51_config
from dronedet.features import load_wave
from dronedet.cnn_v51.identity import standardized_sample_sha256

cfg = load_v51_config(STUDY / 'config/v51.yaml')
def read_rows(path):
    with path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))
train_rows, val_rows = read_rows(FROZEN / 'train.csv'), read_rows(FROZEN / 'val.csv')
inventory_rows = read_rows(INVENTORY)
inventory = {r['source_file_id']: r for r in inventory_rows}
if len(inventory) != len(inventory_rows): raise ValueError('Duplicate source_file_id in inventory')

def resolve(row):
    root = SOURCES[str(row['source'])].resolve()
    rel = str(row['relative_path'])
    if rel.startswith(('/', '\\\\')) or '\\\\' in rel:
        raise ValueError(f'Non-canonical inventory path: {rel!r}')
    path = root.joinpath(*rel.split('/')).resolve()
    path.relative_to(root)
    if not path.is_file(): raise FileNotFoundError(path)
    return path

checked = {}
for row in train_rows + val_rows:
    inv = inventory.get(row['source_file_id'])
    if inv is None: raise ValueError(f"Inventory row missing: {row['source_file_id']}")
    path = resolve(inv)
    checked[inv['source_file_id']] = (inv, path)
print(f'Path preflight passed: {len(checked)} train/validation source files')

# Verify one full-file hash and one standardized clip sample hash for each source.
for source in sorted({r['source'] for r in train_rows + val_rows}):
    row = min((r for r in train_rows + val_rows if r['source'] == source), key=lambda r: r['clip_id'])
    inv, path = checked[row['source_file_id']]
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    if h.hexdigest() != inv['file_sha256']:
        raise ValueError(f"Raw file hash mismatch for source {source}: {path}")
    audio = np.asarray(load_wave(path, cfg.frontend), dtype=np.float32)
    start, count = int(row['start_sample_16k']), int(row['n_samples'])
    clip = np.ascontiguousarray(audio[start:start + count], dtype=np.float32)
    if len(clip) != count or standardized_sample_sha256(clip) != row['sample_sha256']:
        raise ValueError(f"Sample hash mismatch in preflight for {row['clip_id']}")
    print(f'Preflight hash/decode passed: {source}')

In [ ]:
import shlex

cmd = [sys.executable, '-u', str(STUDY / 'scripts/run_cnn_v51_p5_colab.py'),
       '--study-root', str(STUDY),
       '--frozen-attempt', str(FROZEN),
       '--source-inventory', str(INVENTORY),
       '--budget-selection', str(SELECTION),
       '--p4-attempt', str(P4),
       '--output', str(OUT)]
for source, root in SOURCES.items():
    cmd += ['--source-root', f'{source}={root}']

print('Command:', shlex.join(cmd))
import json, platform
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, capture_output=True, text=True).stdout.splitlines()
env_path = OUT.parent / (OUT.name + '.environment.json')
env_path.write_text(json.dumps({'python': sys.version, 'platform': platform.platform(), 'pip_freeze': freeze}, indent=2), encoding='utf-8')
print('Environment record:', env_path)
log_path = OUT.parent / (OUT.name + '.log')
with log_path.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = proc.wait()
print('Exit code:', code, '| log:', log_path)
if code != 0:
    raise RuntimeError(f'P5 runner stopped with exit code {code}; preserve this attempt and inspect its log/gate output.')

## After it finishes

Inspect the printed gate summary and the files in the new `outputs/p5-*` directory. Preserve a failed attempt as-is. These P5 artifacts bind to split-008 and the passing `v51-budget-1g-a5` P4 artifacts. Preserve any P5 failure; P6 can start only after P5 PASS. The historical split-007 remains ineligible. P6 can only start after P5 reports `PASS`; a failed P5 gate must not be bypassed by editing the frozen split or fixed rules. The `.log` and environment record are written alongside the output directory.